# 06 · Model evaluation
Reads the registered model versions: test metrics vs baselines, interval coverage, SHAP and error analysis.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
import plotly.express as px
from src.common.config import get_settings
S = get_settings(); P = S.processed_dir; A = S.analytics_dir

In [ ]:
from src.models.registry import load_json
rows = []
for d in sorted(S.models_dir.iterdir()):
    m = load_json(d.name, 'metrics')
    if m and 'best_model' in m:
        t = m['models'][m['best_model']]['test']
        rows.append({'task': d.name, 'best': m['best_model'], **{k: t.get(k) for k in ('mae','rmse','r2','f1_macro','accuracy')},
                     **{f'baseline_{b}': (v.get('mae') or v.get('f1_macro')) for b, v in m['baselines'].items()}})
pd.DataFrame(rows)

In [ ]:
shap = load_json('speed_h60', 'shap')
pd.DataFrame(shap['mean_abs_shap']).head(15).plot.barh(x='feature', y='value', title='mean |SHAP| · speed +60 min')

In [ ]:
load_json('speed_h60', 'intervals'), load_json('demand_h1', 'intervals')

In [ ]:
err = load_json('speed_h60', 'errors')
pd.DataFrame(err['hour']).sort_values('hour').plot(x='hour', y=['mae','bias'], title='Error by hour')